<h1>Fine-Tuning SAM2 w/ CamVid</h1>

Credits:
- [Video](https://www.youtube.com/watch?v=bcwLbmALyLI)
- [Medium Article](https://medium.com/towards-data-science/train-fine-tune-segment-anything-2-sam-2-in-60-lines-of-code-928dd29a63b3)
- [Repository](https://github.com/sagieppel/fine-tune-train_segment_anything_2_in_60_lines_of_code/tree/main)

In [ ]:
import numpy as np
import torch
import cv2
import os
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import matplotlib.pyplot as plt

In [ ]:
data_dir = "../CamVid/"
data = []

# ff=index, name=filename
for ff, name in enumerate(os.listdir(data_dir + "train/")):
    try:
        data.append({
            "image":data_dir + "train/"+name,
            "annotation":data_dir+"train_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")

for pair in data[:5]:
    print(pair["image"], pair["annotation"]) # to test above works

In [ ]:
def read_batch_OLD(data): # Old version of reading images into masks - read them wrong
     
    # read random image and its annotatio from the CamVid dataset

    entry = data[np.random.randint(len(data))] # Choose a random entry
    img = cv2.imread(entry["image"])
    ann_map = cv2.imread(entry["annotation"])

    # Normally we'd resize here. However, images are smaller than 1024 in both dimensions.
    # Thus, we skip resizing.

    # Fill in missing data
    mat_map = ann_map[:,:,0] # material map
    ves_map = ann_map[:,:,2] # vessel map
    mat_map[mat_map == 0] = ves_map[mat_map == 0] * (mat_map.max() + 1) # merging the two maps together 

    inds = np.unique(mat_map)[1:] # load all indices
    points = []
    masks = []

    # Use unique indices to sort pixels into masks
    for ind in inds:
        mask = (mat_map == ind).astype(np.uint8) # make binary mask
        masks.append(mask)
        coords = np.argwhere(mask > 0) # get all coordinates in mask
        yx = np.array(coords[np.random.randint(len(coords))]) # choose random point/coordinate from mask
        points.append([[yx[1], yx[0]]]) # x,y

    return img, np.array(masks), np.array(points), np.ones([len(masks), 1])

In [ ]:
def disp_np_array_as_img(img:np.array):
    plt.figure(figsize=(10, 10))
    plt.imshow(img)    
    plt.axis('off')
    plt.show()

In [ ]:
def read_batch(data): 
    # read random image and its annotatio from the CamVid dataset

    entry = data[np.random.randint(len(data))] # Choose a random entry
    img = cv2.imread(entry["image"])
    ann_map = cv2.imread(entry["annotation"])

    # Normally we'd resize here. However, images are smaller than 1024 in both dimensions.
    # Thus, we skip resizing.
    all_pixels = ann_map.reshape(-1, 3)
    unique_colors = np.unique(all_pixels, axis=0)
    unique_colors = unique_colors = unique_colors[~np.all(unique_colors == [0,0,0], axis=1)]
    
    # Using the annotation, get all unique colors. Then, sort them into binary masks for each color.
    points = []
    masks = []

    for color in unique_colors:
        binary_mask = np.all(ann_map == color, axis=2).astype(np.uint8) # make binary mask
        # print(binary_mask.shape)
        # print(np.unique(binary_mask))
        mask = np.zeros(shape=(720, 960, 3), dtype=np.uint8)

        # Below binary mask isn't setting??
        mask[binary_mask == 1] = color
        # print(mask[binary_mask == 1].shape)
        masks.append(mask)
        coords = np.argwhere(mask > 0)
        yx = np.array(coords[np.random.randint(len(coords))]) # choose random point/coordinate from mask
        points.append([[yx[1], yx[0]]]) # x,y
        
    return img, np.array(masks), np.array(points), np.ones([len(masks), 1])

if False: read_batch(data) # testing code

In [ ]:
sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda") # load mmodel
predictor = SAM2ImagePredictor(sam2_model) # load net

In [ ]:
predictor.model.sam_mask_decoder.train(True) # enable training of mask decoder
predictor.model.sam_prompt_encoder.train(True) # enable training of prompt decoder

In [ ]:
optimizer = torch.optim.AdamW(
    params=predictor.model.parameters(), 
    lr=1e-5,
    weight_decay=4e-5
)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
def OLD_TRAINING():
    # training loop
    from datetime import datetime

    for itr in range(1, 1001):
        with torch.cuda.amp.autocast(): # cast to mix precision

            image, mask, input_point, input_label = read_batch(data)
            if mask.shape[0] == 0: continue # skip empty batches
            predictor.set_image(image) # apply SAM2 image encoder to training image

            # prompt encoding
            mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
            sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

            # mask decoder
            batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
            high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]
            low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(image_embeddings=predictor._features["image_embed"][-1].unsqueeze(0),image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),sparse_prompt_embeddings=sparse_embeddings,dense_prompt_embeddings=dense_embeddings,multimask_output=True,repeat_image=batched_mode,high_res_features=high_res_features,)
            prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])# Upscale the masks to the original image resolution

            # segmentation loss calculation
            gt_mask = torch.tensor(mask.astype(np.float32)).cuda()
            prd_mask = torch.sigmoid(prd_masks[:, 0])# Turn logit map to probability map
            seg_loss = (-gt_mask * torch.log(prd_mask + 0.00001) - (1 - gt_mask) * torch.log((1 - prd_mask) + 0.00001)).mean() # cross entropy loss
        
            # score loss using IOU
            inter = (gt_mask * (prd_mask > 0.5)).sum(1).sum(1)
            iou = inter / (gt_mask.sum(1).sum(1) + (prd_mask > 0.5).sum(1).sum(1) - inter)
            score_loss = torch.abs(prd_scores[:, 0] - iou).mean()
            loss=seg_loss+score_loss*0.05  # mix losses

            # backpropogate loss
            predictor.model.zero_grad() # empty gradient
            scaler.scale(loss).backward()  # Backpropogate
            scaler.step(optimizer)
            scaler.update() # Mix precision
            
            # Save model
            if itr%1000==0: 
                date_str = str(datetime.now()).replace(":","-")
                torch.save(predictor.model.state_dict(), f"../models/model-{date_str}.torch")
                print("saved model.")
        
            # Display results
            if itr==1: mean_iou=0
            mean_iou = mean_iou * 0.99 + 0.01 * np.mean(iou.cpu().detach().numpy())
            print("step)",itr, "Accuracy(IOU)=",mean_iou)

In [ ]:
def read_image(image_path, mask_path):
    # read an image and its mask
    image = cv2.imread(image_path)[...,::-1] # convert bgr to rgb
    mask = cv2.imread(mask_path, 0) # load masks in grayscale

    # Again, no resizing since images are less than 1024px on both dimensions

    return image, mask

def get_points(mask, num_points): # Sample points inside the input mask
    points = []
    for i in range(num_points):
        coords = np.argwhere(mask > 0)
        yx = np.array(coords[np.random.randint(len(coords))])
        points.append([[yx[1], yx[0]]])
    return np.array(points)

data_dir = "../CamVid/"
val = []
for ff, name in enumerate(os.listdir(data_dir + "val/")[:5]):
    try:
        val.append({
            "image":data_dir + "val/"+name,
            "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")


for pair in val:
    print(pair["image"], pair["annotation"]) # to test above works

I think we're testing it wrong. Our mid-training validation images are just...solid colors?

In [ ]:
# predict masks
def test_predictor(predictor):   
    pair = val[1] # update later to for-loop some images

    image_path, mask_path = pair["image"], pair["annotation"]
    image, mask = read_image(image_path, mask_path)
    input_points = get_points(mask, num_points=30) # arbitrarily get 30 points

    with torch.no_grad():
        predictor.set_image(image.copy())
        masks, scores, logits = predictor.predict(
            point_coords=input_points,
            point_labels=np.ones([input_points.shape[0], 1])
        )

    # short predicted masks from high to low score
    # essentially order masks by confidence scorwa
    np_masks = np.array(masks[:,0]) 
    np_scores = scores[:,0]
    shorted_masks = np_masks[np.argsort(np_scores)][::-1]

    # create empty segmentation map and occupancy map
    # then, add masks to segmentation map one by one
    # we only add a mask if it's consistent with previously added masks
        # AKA less tha 15% overlap with already occupied areas
    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)
    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        # if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        
        # Convert mask to boolean for idnexing
        seg_map[mask.astype(np.uint8)]=i+1
        occupancy_mask[mask > 0.5]=1

    # create colored annotation map
    height, width = seg_map.shape

    # create empty rgb image for colored annotation
    rgb_image = np.zeros([height, width, 3], dtype=np.uint8)

    # map each class to a random color 
    # later, convert this to camvid coloring
    for id_class in range(1, seg_map.max() + 1):
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    
    disp_np_array_as_img(rgb_image)

    disp_np_array_as_img(mask)

    return rgb_image

In [ ]:
def get_uniq_pixels(ann_map):
    all_pixels = ann_map.reshape(-1, 3)
    unique_colors = np.unique(all_pixels, axis=0)
    return unique_colors

### Debug code

In [ ]:
if False: # change to true if debugging as this code kills kernel
    sample=0

    image, mask, input_point, input_label = read_batch(data) # test result
    print("read_batch shapes:", image.shape, mask.shape, input_point.shape, input_label.shape)

    predictor.set_image(image) # apply SAM2 image encoder to training image

    # prompt encoding
    mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
    sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

    # mask decoder
    batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
    high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]
    low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(image_embeddings=predictor._features["image_embed"][-1].unsqueeze(0),image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),sparse_prompt_embeddings=sparse_embeddings,dense_prompt_embeddings=dense_embeddings,multimask_output=True,repeat_image=batched_mode,high_res_features=high_res_features,)
    prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])# Upscale the masks to the original image resolution

    print("predicted mask shape", prd_masks.shape)
    print("gt (ground truth) mask shape", mask.shape)
    
    # Showing masks 
    print("Showing predicted")
    prd_masks = prd_masks.permute(0,2,3,1)
    for pred in prd_masks[:sample]: # cut short to keep notebook scrollable
        pred = pred.cpu().detach().numpy().astype(np.uint8) # puts values between 0 and 255 - RGB values
        disp_np_array_as_img(pred)
        uniq = get_uniq_pixels(pred).astype(np.uint8)
        print(uniq)
        print(len(uniq))

        show_pixels = True
        if show_pixels: print(pred) # OUTPUT IS A SEGMENTED IMAGE WITH MASKS. 


    print("Showing gt")
    for obj in mask[:sample]:
        disp_np_array_as_img(obj)
        print(get_uniq_pixels(obj))

    # ---------------- VALIDATION ----------------

    # 1. Collection predicted masks into one image
    # 2. Apply MSE to images where pixels are non-zero (filler function until Saleh shares code)

    # ground_truth = torch.tensor(np.any(mask > [20, 20, 20], axis=3).astype(np.uint8))
    # # print(ground_truth)
    # print("mask shape:", ground_truth.shape)
    # print("number of nonzero entries:", torch.sum(ground_truth))     

    # predicted = torch.any(prd_masks.to(torch.uint8) < torch.tensor([240, 240, 240]).cuda(), axis=3).to(torch.uint8)
    # # print(predicted)
    # print("predicted shape:", predicted.shape)
    # print("number of nonzero predicted entries:", torch.sum(predicted))

    # from torch import nn
    # criterion = nn.MSELoss()
    # loss = criterion(ground_truth.to(float).cuda(), predicted.to(float).cuda()) # Only seems to work for long types
    # print("loss (MSE):", loss)
    
    gt_mask = torch.tensor(mask.astype(np.float32)).cuda()
    
    prd_mask = torch.sigmoid(prd_masks) # Turn logit map to probability map

    seg_loss = (-gt_mask * torch.log(prd_mask + 0.00001) - (1 - gt_mask) * torch.log((1 - prd_mask) + 0.00001)).mean() # cross entropy loss

    # score loss using IOU
    inter = (gt_mask * (prd_mask > 0.5)).sum(1).sum(1)
    iou = inter / (gt_mask.sum(1).sum(1) + (prd_mask > 0.5).sum(1).sum(1) - inter)
    print(np.mean(iou.cpu().detach().numpy()))
    print(iou.sum())

    # print("prd scores from model (has gradient)", prd_scores, prd_scores.shape)
    print(iou.shape)
    test_score = prd_scores - iou

    # loss1 = torch.abs(prd_scores - iou).sum()
    # loss2 = torch.abs(prd_scores - iou).mean()
    # print(loss1, loss2)
    print("seg loss", seg_loss)

### Currently has a hack solution. Please fix!

In [ ]:
# training loop
from datetime import datetime

sample=1
num_steps = 1e6 # 1 000 000
# num_steps = 10

for itr in range(1, num_steps + 1):
    with torch.cuda.amp.autocast(): # cast to mix precision

        image, mask, input_point, input_label = read_batch(data)
        if mask.shape[0] == 0: continue # skip empty batches
        predictor.set_image(image) # apply SAM2 image encoder to training image

        # prompt encoding
        mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
        sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

        # mask decoder
        batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
        high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]
        low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(image_embeddings=predictor._features["image_embed"][-1].unsqueeze(0),image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),sparse_prompt_embeddings=sparse_embeddings,dense_prompt_embeddings=dense_embeddings,multimask_output=True,repeat_image=batched_mode,high_res_features=high_res_features,)
        prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])# Upscale the masks to the original image resolution

        # print(prd_masks.shape, mask.shape)
        prd_masks = prd_masks.permute(0,2,3,1)
        
        # segmentation loss calculation
        gt_mask = torch.tensor(mask.astype(np.float32)).cuda()
        
        # What is inside gt_mask? Can we just stright compare w/ prd_masks?

        prd_mask = torch.sigmoid(prd_masks) # Turn logit map to probability map

        seg_loss = (-gt_mask * torch.log(prd_mask + 0.00001) - (1 - gt_mask) * torch.log((1 - prd_mask) + 0.00001)).mean() # cross entropy loss

        # backpropogate loss
        predictor.model.zero_grad() # empty gradient        
        scaler.scale(seg_loss).backward()  # Backpropogate - only requirement is that it is a tensor with a gradient
        
        scaler.step(optimizer)
        scaler.update() # Mix precision

        if itr%50==0:
            print("Showing model pred:")
            for pred in prd_mask[:sample]: # sample to keep notebook scrollable
                img = pred.cpu().detach().numpy()
                disp_np_array_as_img(img)
                # print(img.shape)

            print("Showing ground truth:")
            for obj in mask[:sample]:
                disp_np_array_as_img(obj)
                # print(obj.shape)



        # Save model
        if itr%100==0: 
            date_str = str(datetime.now()).replace(":","-")
            torch.save(predictor.model.state_dict(), f"../models/model-{date_str}.torch")
            print("saved model.")
    
        # Display results
        # if itr==1: mean_iou=0
        # detached_iou=np.mean(iou.cpu().detach().numpy())
        # mean_iou = mean_iou * 0.99 + 0.01 * detached_iou

        scaler_seg = seg_loss.cpu().detach()
        print(f"step {itr}) Loss={scaler_seg} Accuracy (Seg Loss)={scaler_seg}")

In [ ]:
import numpy as np
import torch
import cv2
import os
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import matplotlib.pyplot as plt

# Validate our finetuned SAM2

# use bfloat16 for memory efficiency
torch.autocast(device_type="cuda", dtype=torch.float32).__enter__()

# load some example images
data_dir = "../CamVid/"
val = []
for ff, name in enumerate(os.listdir(data_dir + "val/")[:5]):
    try:
        val.append({
            "image":data_dir + "val/"+name,
            "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")


for pair in val:
    print(pair["image"], pair["annotation"]) # to test above works

In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)
input_points = get_points(mask, num_points=30) # arbitrarily get 30 points

In [ ]:
model_dir = "../models/"

sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")

# build finetuned model and load weights
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load(model_dir + os.listdir(model_dir)[-1]))
print("Loaded model:", model_dir + os.listdir(model_dir)[-1])

In [ ]:
# display image

pair = val[1] # update later to for-loop some images

with torch.no_grad():
    predictor.set_image(image.copy())
    masks, scores, logits = predictor.predict(
        point_coords=input_points,
        point_labels=np.ones([input_points.shape[0], 1])
    )

# short predicted masks from high to low score
# essentially order masks by confidence scorwa
np_masks = np.array(masks[:,0]) 
np_scores = scores[:,0]
shorted_masks = np_masks[np.argsort(np_scores)][::-1]

# create empty segmentation map and occupancy map
# then, add masks to segmentation map one by one
# we only add a mask if it's consistent with previously added masks
    # AKA less tha 15% overlap with already occupied areas
seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)
for i in range(shorted_masks.shape[0]):
    mask = shorted_masks[i]
    # if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
    mask[occupancy_mask]=0
    
    # Convert mask to boolean for idnexing
    seg_map[mask.astype(np.uint8)]=i+1
    occupancy_mask[mask > 0.5]=1

# create colored annotation map
height, width = seg_map.shape

# create empty rgb image for colored annotation
rgb_image = np.zeros([height, width, 3], dtype=np.uint8)

# map each class to a random color 
# later, convert this to camvid coloring
for id_class in range(1, seg_map.max() + 1):
    rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

print("number of nonzero entries in predicted:", np.sum(np.any(rgb_image > [0, 0, 0], axis=2)))     

plt.figure(figsize=(10, 10))
plt.imshow(rgb_image)    
plt.axis('off')
plt.show()

plt.figure(figsize=(10, 10))
plt.imshow(image)    
plt.axis('off')
plt.show()